# %% [markdown]
# # STEP 1 — OVERVIEW (Markdown)
# --------------------------------------------------
# This script automates copying heating fuel time-series data for ALL buildings
# in the baseline_models directory.
#
# For each building folder:
# 1. Loads:
#       HEATING/heat_by_fuel.csv
#       TOTAL/energy_results.csv
# 2. Applies a predefined column mapping (source → target)
# 3. Copies data safely (handles mismatched lengths)
# 4. Saves the updated energy_results.csv back to disk
#
# Key features:
# - Fully automated loop over all building folders
# - Skips missing or invalid files gracefully
# - Prints progress and errors for debugging
# - Reusable and scalable for large datasets

In [1]:
# %%
# STEP 2 — IMPORT LIBRARIES
# --------------------------------------------------
import pandas as pd
from pathlib import Path

In [2]:
# %%
# STEP 3 — DEFINE BASE DIRECTORY AND GET BUILDING LIST
# --------------------------------------------------
base_dir = Path("/Users/jakegehrung/Desktop/data_raw/baseline_models")

# Get all building folders (only directories)
building_dirs = [d for d in base_dir.iterdir() if d.is_dir()]

print(f"Found {len(building_dirs)} building folders.")

Found 117 building folders.


In [3]:
# %%
# STEP 4 — DEFINE COLUMN MAPPING
# --------------------------------------------------
column_mapping = {
    "electricity": "electricity_heat",
    "lpg": "lpg_heat",
    "mineral_wood": "mineral_wood_heat",
    "mains_gas": "mains_gas_heat",
    "oil": "oil_heat",
    "wood_logs": "wood_logs_heat",
    "wood_pellets": "wood_pellets_heat",
    "wood_chips": "wood_chips_heat",
    "coal": "coal_heat"
}

print("Column mapping loaded.")

Column mapping loaded.


In [4]:
# %%
# STEP 5 — LOOP THROUGH ALL BUILDINGS
# --------------------------------------------------
success_count = 0
fail_count = 0

for building_dir in building_dirs:
    
    building_id = building_dir.name
    print(f"\nProcessing building: {building_id}")
    
    heating_file = building_dir / "HEATING" / "heat_by_fuel.csv"
    energy_file = building_dir / "TOTAL" / "energy_results.csv"
    
    # Check files exist
    if not heating_file.exists() or not energy_file.exists():
        print("Missing required files. Skipping.")
        fail_count += 1
        continue
    
    try:
        # Load data
        heat_df = pd.read_csv(heating_file)
        energy_df = pd.read_csv(energy_file)
        
        # Validate columns
        missing_source = [col for col in column_mapping if col not in heat_df.columns]
        missing_target = [col for col in column_mapping.values() if col not in energy_df.columns]
        
        if missing_source or missing_target:
            print(f"Missing columns. Source: {missing_source}, Target: {missing_target}")
            fail_count += 1
            continue
        
        # Copy data safely
        for source_col, target_col in column_mapping.items():
            
            source_values = heat_df[source_col].values
            target_length = len(energy_df)
            source_length = len(source_values)
            
            # Initialize column
            energy_df[target_col] = pd.NA
            
            # Safe copy length
            copy_length = min(source_length, target_length)
            
            energy_df.loc[:copy_length-1, target_col] = source_values[:copy_length]
        
        # Save updated file
        energy_df.to_csv(energy_file, index=False)
        
        print("Success")
        success_count += 1
    
    except Exception as e:
        print(f"Error: {e}")
        fail_count += 1

print("\n--- SUMMARY ---")
print(f"Successful: {success_count}")
print(f"Failed: {fail_count}")


Processing building: 1001664924
Success

Processing building: 1001829085
Success

Processing building: 1001063639
Success

Processing building: 1001829071
Success

Processing building: 1234761002
Success

Processing building: 1002539407
Success

Processing building: 1001063637
Success

Processing building: 1001664941
Success

Processing building: 1001991633
Success

Processing building: 1235057414
Success

Processing building: 1001829079
Success

Processing building: 1001664922
Success

Processing building: 1234761003
Success

Processing building: 1001664925
Success

Processing building: 1000238907
Success

Processing building: 1234761004
Success

Processing building: 1002313096
Success

Processing building: 1001870878
Success

Processing building: 1001664940
Success

Processing building: 1234982990
Success

Processing building: 1234806523
Success

Processing building: 1001870876
Success

Processing building: 1001870882
Success

Processing building: 1002143357
Success

Processing buil